# Example 5.4 / 1: A Reproducible Sample

Companion notebook for **5.4 Random Data** in the AMPL lab Documentation, section "A Reproducible Sample". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## A Reproducible Sample

The model generates an integer demand scenario for each product. `Uniform(80, 121)` is
continuous, and `round` maps each draw to an integer from 80 through 121. The two end values are
half as likely as the others, because only half a unit of the range rounds to each of them;
`floor(Uniform(80, 121))` gives 80 through 120 with equal probability. Use a distribution that
matches the quantity and probability model you actually intend.

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;
param simulated_demand {PROD} = round(Uniform(80, 121));

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.read("model.mod")

In [4]:
# data.py
ampl.option["randseed"] = 123
ampl.set["PROD"] = ["bands", "coils", "plate"]

In [5]:
# run.py
for product, value in ampl.param["simulated_demand"].to_dict().items():
    print(f"{product}: {int(value)}")

bands: 117
coils: 84
plate: 107


### Pure AMPL: data.dat and model.run

The same in pure AMPL, with the same `model.mod`:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;

Writing data.dat


In [7]:
%%ampl_eval
reset;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
option randseed 123;
display simulated_demand;

simulated_demand [*] :=
bands  117
coils   84
plate  107
;

